In [ ]:
import json

from palace.mcp_utils.mcp_client import MCPClientPool
from palace.utils.constants import ALOHA_STAGING_URL, GPTJRC_PROD_API_URL
from palace.utils.printing import print
from palace.utils.secrets import ALOHA_STAGING_TOKEN, GPTJRC_PROD_TOKEN

LOCAL_URL = "http://localhost:8080/sse"
# LOCAL_URL = "http://localhost:8090/mcp/sse"
AGENTPOC_URL = "http://localhost:8000/sse"

In [2]:
with MCPClientPool.get_connection(AGENTPOC_URL) as mcp_client:
    tools = mcp_client.list_tools()
    print(tools)

meta=None nextCursor=None tools=[Tool(name='answer_question', title=None, description='Ask questions about the documents stored in the folder\n\nArgs:\nquestion: the question to be answered\n', inputSchema={'properties': {'question': {'description': 'the question to be answered', 'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'answer_questionArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'maxItems': 2, 'minItems': 2, 'prefixItems': [{'type': 'string'}, {'additionalProperties': True, 'type': 'object'}], 'title': 'Result', 'type': 'array'}}, 'required': ['result'], 'title': 'answer_questionOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]


In [30]:
print(list(tools.tools[0].inputSchema["properties"].keys())[0])

question


In [2]:
with MCPClientPool.get_connection(AGENTPOC_URL, ALOHA_STAGING_TOKEN) as mcp_client:
    tools = mcp_client.list_tools()
    print(
        "\n".join(
            [
                f"{tool.name}\n{tool.description}\n"
                + json.dumps(
                    {
                        k: v["description"]
                        for k, v in tool.inputSchema["properties"].items()
                    },
                    indent=4,
                )
                + "\n"
                for tool in tools.tools
            ]
        )
    )

answer_question
Ask questions about the documents stored in the folder

Args:
question: the question to be answered

{
    "question": "the question to be answered"
}



In [8]:
with MCPClientPool.get_connection(AGENTPOC_URL) as mcp_client:
    response = mcp_client.call_tool(
        "answer_question", {"question": "What is the GDPR?"}
    )
print(f"[bold]Result:[/]\n{response.content[0].text}")
print(
    f"\n[bold]Metrics:[/]\n{json.dumps(response.structuredContent['result'][1], indent=4)}"
)

Result:
The **General Data Protection Regulation (GDPR)** is a European Union regulation that establishes a comprehensive legal framework for the protection of personal data of individuals within the EU (and, by extension, anyone whose data is processed by entities operating in the EU).  

### Key points of the GDPR  

| Aspect | What it means |
|--------|----------------|
| **Scope and applicability** | Applies to any organization—whether based inside or outside the EU—that processes the personal data of individuals located in the EU. |
| **Personal data definition** | Any information that can directly or indirectly identify a natural person (e.g., name, ID number, location data, online identifiers, health information, etc.). |
| **Lawful bases for processing** | Processing is permitted only when at least one of the following applies: consent, performance of a contract, legal obligation, vital interests, public task, or legitimate interests (balanced against the data subject’s rights)

In [ ]:
with MCPClientPool.get_connection(ALOHA_STAGING_URL, ALOHA_STAGING_TOKEN) as mcp_client:
    response = mcp_client.call_tool(
        "scopus_search_query",
        {
            "full_keywords": """Provide the exact answer, without any additional text (for example, if the answer is a name, write only the name as it is):
Who received the IEEE Frank Rosenblatt Award in 2010?"""
        },
    )
print(response)

In [ ]:
research_topics = {
    "physics": [
        "Quantum computing",
        "Dark matter",
        "Gravitational waves",
        "Nanophotonics",
        "Superconductors",
    ],
    "engineering": [
        "Renewable energy",
        "Robotics",
        "Smart cities",
        "Biomedical devices",
        "Additive manufacturing",
    ],
    "statistics": [
        "Bayesian inference",
        "Big data analytics",
        "Causal inference",
        "Machine learning optimization",
        "Statistical genomics",
    ],
    "computer science": [
        "Artificial intelligence ethics",
        "Quantum algorithms",
        "Cybersecurity threat detection",
        "Natural language processing",
        "Blockchain",
    ],
    "biology": [
        "CRISPR",
        "Microbiome and human health",
        "Climate change",
        "Cancer immunotherapy",
        "Synthetic biology",
    ],
    "mathematics": [
        "Topology",
        "Cryptography",
        "Mathematical modeling in epidemiology",
        "Algebraic geometry",
        "Numberical analysis",
    ],
    "chemistry": [
        "Catalysts for green chemistry",
        "Drug design",
        "Materials chemistry for energy storage",
        "Chemical sensors and biosensors",
        "Photochemistry",
    ],
}
papers = {
    subject: {field: [] for field in fields}
    for subject, fields in research_topics.items()
}
print(json.dumps(papers, indent=4))

In [ ]:
with MCPClientPool.get_connection(ALOHA_STAGING_URL, ALOHA_STAGING_TOKEN) as mcp_client:
    response = mcp_client.call_tool(
        "scopus_search_query",
        {
            "api_query": "test",
            "return_fields": ["title", "doi", "date", "authors"],
        },
    )
    print(response)

In [ ]:
from palace.models.openai_compatible_model import OpenAICompatibleModel
from palace.utils.printing import loading

model = OpenAICompatibleModel(GPTJRC_PROD_API_URL, GPTJRC_PROD_TOKEN, "openai/gpt-4o")

with MCPClientPool.get_connection(ALOHA_STAGING_URL, ALOHA_STAGING_TOKEN) as mcp_client:
    with loading():
        for subject, subject_papers in papers.items():
            for field, field_papers in subject_papers.items():
                for p, paper in enumerate(field_papers):
                    try:
                        response = mcp_client.call_tool(
                            "get_paper_text_from_doi",
                            {"doi": paper["doi"]},
                        )
                        full_text = response.content[0].text[:300000]
                        generated_text = model.generate(
                            [
                                {
                                    "role": "system",
                                    "content": """You will be given the full text of a scientific paper (in XML format but try to extract the relevant text parts), and will be prompted to generate a question pair that requires the knowledge of that paper in order to be answered. The generated question must be such that it cannot be answered without knowing the content of the provided paper, but it should also be general and self-contained enough on its own, without making references to the paper. For instance, it can't contain stuff like 'how do the authors ...'. For instance, this question would not be allowed: 'What specific technique do the authors suggest could enable quantum advantage in genetic merit prediction beyond raw matrix acceleration?'. An example of allowed question is: 'What are three distinct contributions that Hamiltonian simulation could make to improving membrane-based resource recovery from agricultural waste streams?' or 'Which quantum algorithm is considered suitable for solving low-rank linear systems specifically in the context of estimating genetic merits in large-scale animal breeding?'. After the question, also include the answer to that question. Make the answer as tight and blunt as possible. If possible, generate questions that have a definitive and blunt answer. For instance, for question 'Which quantum algorithm is considered suitable for solving low-rank linear systems specifically in the context of estimating genetic merits in large-scale animal breeding?', the answer should be something like 'HHL algorithm'. Try to avoid verbose answers. Your response must consist of exactly two lines of text: one with the question, and one with the answer, such as:
```<question here>
<answer here>```
Nothing else can be in your response.""",
                                },
                                {
                                    "role": "user",
                                    "content": f"Write a question-answer pair that requires information contained in the following paper in order to be answered.\n\n\n{full_text}",
                                },
                            ]
                        )
                        print(generated_text)
                        print()
                        question, answer = generated_text.split("\n")
                        papers[subject][field][p]["question"] = question
                        papers[subject][field][p]["answer"] = answer

                    except Exception as e:
                        print(f"There was an error for paper {paper}:\n{e}")
                        continue

print(json.dumps(papers, indent=4))

In [ ]:
with open("scopus_tasks.json", "w") as f:
    json.dump(papers, f, indent=4)

In [ ]:
from palace.utils.printing import loading

with MCPClientPool.get_connection(ALOHA_STAGING_URL, ALOHA_STAGING_TOKEN) as mcp_client:
    with loading():
        response = mcp_client.call_tool(
            "get_paper_text_from_doi",
            {"doi": "10.1016/j.omtm.2025.101493"},
        )

print(response)
print(response.content[0].text)
